<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_GeoPolitical_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

# MOS :: Middle East Oil Security
### An LLM-Based Geopolitical What-If Simulator

> **Action → Consequence → Reaction → Consequence → ...**

**MOS** explores how geopolitical situations may evolve when strategic actors respond to one another in an uncertain and changing world.

**Chanakya** proposes strategic alternatives.  
**Centaur** evaluates their relative likelihood.  
**Shakuni** -- the dice roller -- selects a path under uncertainty.  
**ZeitWorld** translates the selected Event into the next WorldState.

The cycle repeats, creating one possible **Scenario** from an initial hypothetical Trigger.

> **The objective is not to predict the future, but to explore plausible, path-dependent futures arising from strategic choices.**

In [1]:
from IPython.display import Markdown, display
from datetime import datetime
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))

ॐ श्री सरस्वत्यै नमः 2026-10-04 09:03:10.865869+05:30


In [2]:
!pip install -q python-docx
#from docx.oxml import OxmlElement
#from docx.oxml.ns import qn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.3 MB/s eta 0:00:00


In [3]:
!wget -O CentaurLogo.png -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/logo-S.png

!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py
!wget -O mosReporter.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosReporter.py
!wget -O mosDOCx.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosDOCx.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *
from mosDOCx import *

In [24]:
import importlib

import mos
import mosZeitWorld
import mosChanakya
import mosCentaur
import mosReporter
import mosDOCx

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)
importlib.reload(mosCentaur)
importlib.reload(mosReporter)
importlib.reload(mosDOCx)

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *

## Initialise Runtime Environment

Authenticate OpenAI, select the working LLM, mount Google Drive and define the paths to the MOS data, roles and prompts.

The directory listings provide a quick check that all required resources are available.

In [5]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Mounted at /content/drive


## Get World Facts

In [6]:
# Build WorldFactBase
#
# Static PDF-derived Markdown files and live Moneycontrol news
# are assembled into a single corpus used to initialise W0.
# No real-world information enters the simulation after W0.
# ------------------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)

Static sources : 8
Live sources   : 1
Characters     : 235,072
Words          : 35,600


### Defining the World

In [7]:
# Define the WorldState variables -- SET A
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------

worldVariables_SetA = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [7]:
# Define the WorldState variables  -- SET B
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------
worldVariables_SetB = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction=(
            "Higher index means higher oil price."
        )
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction=(
            "Higher index means greater oil flow."
        )
    ),

    WorldVariable(
        name="Regional Diplomatic Engagement",
        description=(
            "Degree of active diplomatic engagement aimed at managing, "
            "containing or resolving the regional crisis, including "
            "direct negotiations, mediation, backchannels and "
            "de-escalation mechanisms."
        ),
        direction=(
            "Higher index means greater diplomatic engagement "
            "and stronger functioning channels for de-escalation."
        )
    ),

    WorldVariable(
        name="Economic Pressure on Iran",
        description=(
            "Degree of economic pressure on Iran arising from sanctions, "
            "trade and financial restrictions, blockade measures, "
            "export constraints and related economic coercion."
        ),
        direction=(
            "Higher index means greater economic pressure on Iran."
        )
    ),

    WorldVariable(
        name="US/Regional Partner Cohesion",
        description=(
            "Degree of alignment between the United States and key "
            "regional partners, including Israel, Saudi Arabia and "
            "other Gulf states, in responding to Iran and the "
            "regional crisis."
        ),
        direction=(
            "Higher index means greater alignment and willingness "
            "to coordinate with US policy."
        )
    ),

    WorldVariable(
        name="Iran Domestic Stability",
        description=(
            "Degree of stability and resilience within Iran's political, "
            "military, security and domestic system under the pressures "
            "generated by the regional crisis."
        ),
        direction=(
            "Higher index means greater Iranian domestic stability."
        )
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction=(
            "Higher index means greater US-Iran tension."
        )
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )

]

In [8]:
worldVariables = worldVariables_SetB

## Agents and their Roles

### ZeitWorld

In [9]:
# Load ZeitWorld roles and prompts
#
# ZW_Role                  : defines ZeitWorld's role and discipline
# ZW_Initialisation_Prompt : creates W0 from the WorldFactBase
# ZW_Update_Prompt         : evolves the current WorldState after an Event
# ------------------------------------------------------------

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")
ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [10]:
# Instantiate ZeitWorld
#
# ZeitWorld is created with its LLM client, model and governing role.
# The initialised flag confirms whether W0 has already been created.
# ------------------------------------------------------------

ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
#print(ZeitWorld.initialised)
#print(ZeitWorld.model)

### Chanakya

In [11]:
# Load Chanakya roles and strategy
#
# Generic Role : defines the common strategic discipline of all Chanakyas
# US Role      : defines US objectives, interests and constraints
# Iran Role    : defines Iranian objectives, interests and constraints
# BinaryHD     : generates two contrasting Hawk and Dove candidates
# ============================================================

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [12]:
# Instantiate Chanakya agents
#
# Each Chanakya represents a national actor with its own
# interests and constraints, using the current Binary Hawk/Dove strategy.
# The registry maps each actor to its corresponding Chanakya.
# ============================================================

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Centaur

In [13]:
# Load Centaur roles
#
# Generic Role : defines the common adjudication discipline of all Centaurs
# UNSG Role    : provides a fair, independent UN diplomatic perspective
# ============================================================

CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

In [14]:
# Instantiate Centaur adjudicator
#
# CentaurUNSG evaluates Chanakya's Candidate Events from the
# perspective of a fair and independent UN Secretary-General.
# ============================================================
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

## Initialisation of Scenario

In [15]:
# Initialise Simulation State
#
# W stores the evolving WorldStates of the simulation.
# E stores the sequence of Events defining the Scenario.
# W0_Template defines the empty structure from which W0 is created.
# ------------------------------------------------------------

E = []
W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

# Initialise the first WorldState
#
# ZeitWorld uses the WorldFactBase and W0 template to create W0.
# The completed WorldState is automatically appended to W as W[0].
# ============================================================

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

#print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 57,108 | Output: 2,048 | Reasoning: 45 | Total: 59,156


In [ ]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

In [16]:
# Define initial Trigger options
#
# Two alternative Events are available to start the simulation.
# One is manually selected as E0; thereafter Events are generated
# and selected by the simulation.
# ============================================================

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for US"
)


In [17]:
# Select the initial Trigger
#
# The manually selected DefaultTrigger starts the first epoch.
# Its selection weight is None because it was not chosen by Shakuni.
# ============================================================
DefaultTrigger = Trigger1
#DefaultTrigger = Trigger2
DefaultTrigger.show()
print(DefaultTrigger.selection_weight)


Event     : 
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran
selection_weight : None
None


In [18]:
# Initialise / Reset Scenario
#
# Retains W0 and clears all subsequent WorldStates and Events,
# restoring the simulation to its starting condition.
# ============================================================

def resetWorld():
    del W[1:]
    E.clear()

    print("Scenario reset to W0.")

# ------------------------------------------------------------

resetWorld()

Scenario reset to W0.


### Define Single Epoch

In [19]:
def runEpoch():
    # ============================================================
    # EPOCH STEP 1 — Establish Current Trigger
    #
    # The epoch is derived from the current WorldState history.
    # For the first epoch, DefaultTrigger becomes E0.
    # Thereafter, the last Reaction becomes the next Trigger.
    # ============================================================

    epoch = len(W) - 1
    if len(E) == 0:
        DefaultTrigger.event_id = f"E{epoch}"
        E.append(DefaultTrigger)
    else:
        assert E[epoch] is E[-1]

    #E[epoch].show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 1 complete" )

    # ============================================================
    # EPOCH STEP 2 — Update WorldState
    #
    # ZeitWorld applies Trigger E[epoch] to W[epoch] and creates
    # the next WorldState W[epoch+1].
    # ============================================================

    _ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

    #E[epoch].show()
    #W[epoch+1].show(-2)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 2 complete" )

    # ============================================================
    # EPOCH STEP 3 — Select Active Actor
    #
    # The Turn Controller identifies which actor responds to the
    # Current Trigger and activates its corresponding Chanakya.
    # ============================================================

    #ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)
    ActiveChanakya = getActiveChanakya(E[epoch], ChanakyaRegistry)
    #ActiveChanakya.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 3 complete" )

    # ============================================================
    # EPOCH STEP 4 — Generate Candidate Reactions
    #
    # The Active Chanakya evaluates the updated WorldState and
    # generates Candidate Events as possible reactions to the Trigger.
    # ============================================================

    Candidates = ActiveChanakya.propose(
        client=client,
        model=model,
        trigger=E[epoch],
        world=W[epoch+1]
    )

    #for i, candidate in enumerate(Candidates):
    #    print(f"\nCandidate Reaction {i+1}")
    #    print("-" * 60)
    #    candidate.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 4 complete" )

    # ============================================================
    # EPOCH STEP 5 — Adjudicate Candidate Reactions
    #
    # Centaur evaluates the Candidate Events against the current
    # WorldState and assigns each a relative simulation weight.
    # ============================================================

    weights = Centaur_UNSG.evaluate(
        client,
        model,
        E[epoch],
        W[epoch + 1],
        Candidates
    )

    #print("\nCentaur Weights")
    #print("----------------")
    print("Weights",weights)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 5 complete" )

    # ============================================================
    #
    # Centaur weights represent the relative conditional likelihood
    # of each Candidate at this decision point.
    #
    # Shakuni makes a single stochastic selection using these weights.
    # The selected Candidate becomes the Reaction, retains its Centaur
    # weight as selection_weight, receives the next Event ID, and is
    # appended to E as the Trigger for the next epoch.
    # ============================================================

    Reaction = Shakuni(Candidates, weights)

    Reaction.event_id = f"E{len(E)}"
    E.append(Reaction)

    #print("\nShakuni has spoken")
    #print("------------------")
    #Reaction.show()
    #print(Reaction.selection_weight)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 6 complete" )
    print("Epoch ends ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")



##Scenario | Multiple Epochs

In [20]:
MaxEvents = 4
_scenarioID = "TR1ME4"

resetWorld()

while len(E) < MaxEvents:
    runEpoch()

# Apply the final Event and close the Scenario
_ = ZeitWorld.update(W, E[-1], ZW_Update_Prompt)

# Structural integrity check
assert len(E) == MaxEvents
assert len(W) == MaxEvents + 1
assert len(W) == len(E) + 1

print("\nScenario complete")
print("-----------------")
print("Events      :", len(E))
print("WorldStates :", len(W))

Scenario reset to W0.
epoch : 0 len(W): 1 len(E): 1  ***** Step # 1 complete
Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Input: 3,320 | Output: 2,585 | Reasoning: 237 | Total: 5,905
Appended  WorldState ---------- W1
epoch : 0 len(W): 2 len(E): 1  ***** Step # 2 complete
epoch : 0 len(W): 2 len(E): 1  ***** Step # 3 complete
Chanakya working ---------- ChanakyaUS_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,483 | Output: 312 | Reasoning: 77 | Total: 2,795
epoch : 0 len(W): 2 len(E): 1  ***** Step # 4 complete
Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Input: 2,345 | Output: 80 | Reasoning: 64 | Total: 2,425
Weights [0.78, 0.22]
epoch : 0 len(W): 2 len(E): 1  ***** Step # 5 complete
epoch : 0 len(W): 2 len(E): 2  ***** Step # 6 complete
Epoch ends ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
epoch : 1 len(W): 2 len(E): 2  ***** Step # 1 complete
Updating WorldState ---------- W1
Model: gpt-5.6

## Reporter

In [21]:
MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
#WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
#ZW_PATH = f"{MOS_HOME}/ZeitWorld"
#CH_PATH = f"{MOS_HOME}/Chanakya"
#CT_PATH = f"{MOS_HOME}/Centaur"
PR_PATH = f"{MOS_HOME}/Reporter"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"
#!ls -lh "{PR_PATH}"

In [22]:
# ============================================================
# Load PressReporter Roles and Prompts
#
# Generic Role defines reporting discipline.
# Narrative Prompt produces the human-readable Scenario history.
# Structured Prompt produces the machine-oriented Markdown record.
# ============================================================

PR_Generic_Role = getPrompt(f"{PR_PATH}/PR_Generic_Role_v0.txt")

#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_v0.txt")
#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v2.txt")
PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v3.txt")

PR_Structured   = getPrompt(f"{PR_PATH}/PR_Structured_v0.txt")


# ============================================================
# Instantiate Human Reporter
# ============================================================

Reporter = Reporter_Agent(
    client=client,
    model=model,
    agent_id="Vyasa",
    generic_role=PR_Generic_Role
)

In [ ]:
#ScenarioText = PressReporter._scenarioText(W, E)
#print(ScenarioText)

In [25]:
# ============================================================
# Generate Narrative Scenario Report
#
# PressReporter converts the completed Scenario (W + E) into a
# human-readable narrative using the Narrative reporting prompt.
# The resulting report is then formatted and saved as a Word document.
# ============================================================
NarrativeReport = Reporter.GenerateReport(
    W,
    E,
    PR_Narrative
)

#print(NarrativeReport)

NarrativeFile = createNarrativeDocx(
    NarrativeReport,
    W,
    E,
    scenarioID = _scenarioID,
    title="Middle East Oil Security — Scenario Brief"
)

Reporter working ---------- Vyasa | Narrative
Model: gpt-5.6-luna | Effort: medium | Input: 12,115 | Output: 966 | Reasoning: 198 | Total: 13,081
Word report created: /content/MOS_TR1ME4_261004_092057.docx


In [37]:
# ============================================================
# Generate Structured Scenario Report
#
# Produces a structured Markdown representation of the completed
# Scenario for subsequent machine-based analysis.
#
# ============================================================

StructuredReport = Reporter.GenerateReport(
    W,
    E,
    PR_Structured
)

display(Markdown(StructuredReport))

Reporter working ---------- Vyasa | Narrative
Model: gpt-5.6-luna | Effort: medium | Input: 8,278 | Output: 6,665 | Reasoning: 155 | Total: 14,943


# Scenario Summary

- **Number of Events:** 3
- **Number of WorldStates:** 4
- **Initial WorldState:** W0
- **Terminal WorldState:** W3

## WorldState Sequence

W0 → **E0** → W1 → **E1** → W2 → **E2** → W3

---

# Event E0

## Event Information

- **Event ID:** E0
- **Actor:** Iran
- **Action:** Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
- **Rationale:** Initial Trigger for Iran
- **Selection Weight:** None
- **Source WorldState:** W0
- **Resulting WorldState:** W1

## Resulting WorldState: W1

### Variable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 115
- **Index Change:** 100 → 115
- **Status:** Brent crude prices rise further as Iran's declared closure of the Strait of Hormuz and attacks on U.S. naval forces intensify the existing Gulf supply and shipping disruption.
- **Rationale:** The index rises from 100 to 115 because the event directly threatens a major oil-transit route already affected by more than 10 million barrels per day of shut-in Gulf output, depleted inventories and maritime disruption. The additional closure order and attacks justify a substantial but not extreme increase above the already elevated baseline condition.
- **Conditions:**
  - More than 10 million barrels per day of Gulf output remained shut in during August, with recovery deferred.
  - Global observed oil inventories have fallen substantially since February, reducing available market buffers.
  - Strait of Hormuz and Bab el-Mandeb disruptions continue to constrain exports and increase shipping costs.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.

### Variable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 65
- **Index Change:** 100 → 65
- **Status:** Oil and petroleum-product flows through the Strait of Hormuz are reduced to a severely restricted level. Iran's closure order and efforts to prevent tanker transit further limit the previously existing escorted, bypass and workaround movements.
- **Rationale:** The index falls from 100 to 65 because the event directly orders the prevention of commercial tanker transit, worsening a condition that was already severely constrained by Iranian attacks, U.S. naval operations and limited workarounds. Some residual movement may remain through escorts, alternative routes or transfers, so the flow is not assessed as zero.
- **Conditions:**
  - Iranian threats, attacks and the declared closure of the Strait remain operative.
  - Iranian forces have been ordered to prevent commercial tanker transit.
  - A U.S. naval blockade and military escorts affect shipping through and around the Strait.
  - Alternative pipelines, overland routes and ship-to-ship transfers are partially offsetting the disruption.

### Variable: India Gulf Oil Imports

- **Variable Name:** India Gulf Oil Imports
- **Index:** 92
- **Index Change:** 100 → 92
- **Status:** India's imports of Gulf-origin crude are further constrained by the declared closure of the Strait of Hormuz and attacks on naval forces. India continues to use diversified and non-Hormuz sourcing, but the reliability and volume of Gulf-linked imports are reduced.
- **Rationale:** The index falls from 100 to 92 because the event directly restricts the principal maritime route affecting Gulf-origin supplies, adding to existing reliability problems. India's prior diversification, sourcing from approximately 40 countries and use of alternative routes moderate the decline rather than eliminating it.
- **Conditions:**
  - India remains dependent on imported oil, with reported oil-and-gas import dependency of approximately 88.7%.
  - India is sourcing crude from approximately 40 countries and using alternative maritime routes.
  - Hormuz and Gulf shipping disruptions continue to affect the reliability and composition of Indian crude supplies.
  - Iran's closure order and tanker-prevention operations further restrict Gulf-origin crude reaching India through Hormuz.

### Variable: Iran Sanctions Enforcement

- **Variable Name:** Iran Sanctions Enforcement
- **Index:** 100
- **Index Change:** 100 → 100
- **Status:** Sanctions enforcement remains extensive but is not directly altered by Iran's operational closure of the Strait and attacks on U.S. naval forces. Existing enforcement pressure and Iranian circumvention networks remain operative.
- **Rationale:** The index remains at 100 because the event changes maritime and military conditions but does not itself establish a change in the effectiveness of sanctions enforcement. The existing U.S. blockade, expanded sanctions campaign and Iranian shadow-fleet and financial networks continue to define this variable.
- **Conditions:**
  - The U.S. blockade of Iranian ports remains in place.
  - The United States is conducting an expanded sanctions campaign targeting oil, shipping, finance and additional economic sectors.
  - Iran continues to rely on shadow-fleet and Russia-linked financial networks to facilitate trade and sanctions evasion.

### Variable: US/Allied Cohesion

- **Variable Name:** US/Allied Cohesion
- **Index:** 104
- **Index Change:** 100 → 104
- **Status:** The Iranian closure declaration and attacks on U.S. naval forces create an immediate common security pressure for the United States and cooperating states. Alignment remains incomplete because partners continue to differ over military access, exposure to retaliation, sanctions and diplomacy.
- **Rationale:** The index rises from 100 to 104 because a direct Iranian challenge to commercial shipping and U.S. naval forces provides additional grounds for coordinated security responses among existing partners. The increase is limited by the operative disagreements over access, retaliation risk, sanctions and diplomatic policy.
- **Conditions:**
  - U.S. and Israeli military operations and strategic objectives remain linked.
  - Gulf states face continuing Iranian or Iran-linked attacks and are assessing security cooperation with the United States.
  - Regional governments are pursuing parallel diplomatic engagement while weighing the costs of confrontation.
  - Iran's closure of Hormuz and attacks on U.S. naval forces create additional immediate security pressure on the United States and cooperating states.

### Variable: Iran Internal Cohesion

- **Variable Name:** Iran Internal Cohesion
- **Index:** 100
- **Index Change:** 100 → 100
- **Status:** Iran's governing and security apparatus has executed a unified closure and attack order, but the event provides no direct evidence that the underlying political divisions, leadership uncertainty or IRGC influence have materially changed.
- **Rationale:** The index remains at 100 because execution of the stated military action demonstrates operational implementation but does not by itself establish a broader change in internal political cohesion. Existing divisions over negotiations, leadership uncertainty, IRGC influence and repression remain operative.
- **Conditions:**
  - Mojtaba Khamenei remains Supreme Leader but has not appeared publicly since his appointment.
  - Iranian officials remain divided over negotiations, sanctions relief and the conditions for reopening Hormuz.
  - Security forces continue arrests and repression following the 2026 protests and wartime allegations of collaboration.
  - Iranian forces have carried out the declared order to close Hormuz and attack U.S. naval forces, without establishing that broader internal divisions have been resolved.

### Variable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 115
- **Index Change:** 100 → 115
- **Status:** US-Iran tension increases to an even more acute level as Iran declares Hormuz closed, attempts to prevent tanker transit and attacks U.S. naval forces amid an existing blockade, sanctions campaign and stalled diplomacy.
- **Rationale:** The index rises from 100 to 115 because the event adds direct Iranian attacks on U.S. military forces and an explicit challenge to commercial navigation to the already active blockade, sanctions and limited strikes. The magnitude reflects a major escalation in military confrontation without asserting a broader change beyond the event.
- **Conditions:**
  - U.S. and Iranian forces continue exchanging limited strikes.
  - The U.S. blockade and expanded sanctions campaign remain active.
  - Diplomatic contacts have stalled, with competing accounts of possible nuclear-inspection proposals and the status of negotiations.
  - Iran has declared Hormuz closed, ordered the prevention of tanker transit and attacked U.S. naval forces operating in the Gulf.

### Variable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 110
- **Index Change:** 100 → 110
- **Status:** Regional armed conflict intensifies in the Gulf as Iranian forces attack U.S. naval forces while attempting to enforce closure of the Strait. Existing fighting in Lebanon, Yemen, Iraq and other theaters remains operative.
- **Rationale:** The index rises from 100 to 110 because the event adds direct kinetic attacks on U.S. naval forces and increases military activity around a strategically important maritime chokepoint. The increase is significant but limited because the event does not itself establish new fighting in every existing theater.
- **Conditions:**
  - U.S.-Iran hostilities and attacks on shipping around Hormuz remain active, with Iran now declaring the Strait closed and attacking U.S. naval forces.
  - Israeli operations and Hezbollah-related conflict continue in Lebanon.
  - Houthi attacks and Saudi responses continue around Yemen and the Bab el-Mandeb.
  - Iran-aligned groups and regional forces remain involved in attacks affecting Iraq and Gulf energy infrastructure.

---

# Event E1

## Event Information

- **Event ID:** E1
- **Actor:** US
- **Action:** Expand military operations to suppress Iran's ability to enforce the closure, including strikes against Iranian maritime, missile and command assets involved in attacks on U.S. forces and tanker interdiction, while intensifying coalition naval operations to reopen and secure the Strait.
- **Rationale:** A forceful response could restore deterrence, protect U.S. forces and commercial shipping, impose immediate costs for the attacks, and demonstrate that Iran cannot gain strategic leverage by closing Hormuz.
- **Selection Weight:** 0.8
- **Source WorldState:** W1
- **Resulting WorldState:** W2

## Resulting WorldState: W2

### Variable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 120
- **Index Change:** 115 → 120
- **Status:** Brent crude prices rise further as expanded U.S. strikes and intensified coalition operations add immediate military risk around the already closed Strait of Hormuz and Gulf shipping routes.
- **Rationale:** The index rises from 115 to 120 because the event expands direct military operations around the principal oil-transit chokepoint while more than 10 million barrels per day of Gulf output remain shut in and inventories are depleted. The increase is substantial but limited because the operations are also intended to restore navigation rather than establish a permanent additional supply loss.
- **Conditions:**
  - More than 10 million barrels per day of Gulf output remained shut in during August, with recovery deferred.
  - Global observed oil inventories have fallen substantially since February, reducing available market buffers.
  - Strait of Hormuz and Bab el-Mandeb disruptions continue to constrain exports and increase shipping costs.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - Expanded U.S. strikes and coalition naval operations are creating additional immediate military risk around the Strait and Gulf shipping.

### Variable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 62
- **Index Change:** 65 → 62
- **Status:** Oil and petroleum-product flows through the Strait of Hormuz fall to an even more restricted level during expanded U.S. strikes and intensified coalition operations. The operations may support eventual reopening, but the immediate combat environment further impedes commercial transit.
- **Rationale:** The index falls from 65 to 62 because expanded strikes against maritime, missile and command assets involved in tanker interdiction increase immediate combat hazards around a route already subject to Iran's closure order and U.S. naval operations. The decline is modest because intensified coalition operations also preserve some potential for escorted or controlled movement and eventual reopening.
- **Conditions:**
  - Iranian threats, attacks and the declared closure of the Strait remain operative.
  - Iranian forces have been ordered to prevent commercial tanker transit.
  - A U.S. naval blockade and military escorts affect shipping through and around the Strait.
  - Alternative pipelines, overland routes and ship-to-ship transfers are partially offsetting the disruption.
  - Expanded U.S. strikes target Iranian maritime, missile and command assets involved in tanker interdiction.
  - Coalition naval operations have intensified to reopen and secure the Strait.

### Variable: India Gulf Oil Imports

- **Variable Name:** India Gulf Oil Imports
- **Index:** 90
- **Index Change:** 92 → 90
- **Status:** India's Gulf-origin crude imports are further constrained by intensified military operations around the closed Strait of Hormuz. Diversified and non-Hormuz sourcing continues to moderate, but does not eliminate, the reduction in Gulf-linked supply reliability.
- **Rationale:** The index falls from 92 to 90 because expanded strikes and intensified naval operations add immediate transit risk to an already restricted route, further reducing the reliability of Gulf-origin crude reaching India. India's sourcing from approximately 40 countries and use of alternative routes limit the magnitude of the decline.
- **Conditions:**
  - India remains dependent on imported oil, with reported oil-and-gas import dependency of approximately 88.7%.
  - India is sourcing crude from approximately 40 countries and using alternative maritime routes.
  - Hormuz and Gulf shipping disruptions continue to affect the reliability and composition of Indian crude supplies.
  - Iran's closure order and tanker-prevention operations further restrict Gulf-origin crude reaching India through Hormuz.
  - Expanded U.S. and coalition military operations create additional short-term hazards for Gulf-origin shipping.

### Variable: Iran Sanctions Enforcement

- **Variable Name:** Iran Sanctions Enforcement
- **Index:** 100
- **Index Change:** 100 → 100
- **Status:** Sanctions enforcement remains extensive but is not directly changed by the expansion of military operations against Iranian maritime, missile and command assets. Existing enforcement pressure and Iranian circumvention networks remain operative.
- **Rationale:** The index remains at 100 because the event concerns military suppression of Iran's maritime and military capabilities rather than a change in sanctions authorities, financial enforcement or the effectiveness of Iranian circumvention networks. The existing blockade, sanctions campaign and shadow-fleet and financial networks therefore remain the relevant conditions.
- **Conditions:**
  - The U.S. blockade of Iranian ports remains in place.
  - The United States is conducting an expanded sanctions campaign targeting oil, shipping, finance and additional economic sectors.
  - Iran continues to rely on shadow-fleet and Russia-linked financial networks to facilitate trade and sanctions evasion.

### Variable: US/Allied Cohesion

- **Variable Name:** US/Allied Cohesion
- **Index:** 108
- **Index Change:** 104 → 108
- **Status:** U.S. and cooperating states intensify naval coordination to suppress Iran's interdiction efforts and secure the Strait. Existing differences over military access, exposure to retaliation, sanctions and diplomacy remain, limiting the increase in cohesion.
- **Rationale:** The index rises from 104 to 108 because the event involves intensified coalition naval operations with a shared operational purpose: reopening and securing the Strait against Iranian interdiction. The increase is limited because the event does not resolve the existing disagreements over access, retaliation risk, sanctions and diplomatic policy.
- **Conditions:**
  - U.S. and Israeli military operations and strategic objectives remain linked.
  - Gulf states face continuing Iranian or Iran-linked attacks and are assessing security cooperation with the United States.
  - Regional governments are pursuing parallel diplomatic engagement while weighing the costs of confrontation.
  - Iran's closure of Hormuz and attacks on U.S. naval forces create additional immediate security pressure on the United States and cooperating states.
  - Coalition naval operations have intensified to suppress Iranian tanker interdiction and reopen and secure the Strait.

### Variable: Iran Internal Cohesion

- **Variable Name:** Iran Internal Cohesion
- **Index:** 100
- **Index Change:** 100 → 100
- **Status:** Iran's governing and security apparatus remains engaged in enforcing the closure while facing expanded U.S. strikes. The event provides no direct evidence that underlying political divisions, leadership uncertainty or IRGC influence have materially changed.
- **Rationale:** The index remains at 100 because strikes against Iranian military and command assets establish external pressure and possible operational disruption, but do not by themselves demonstrate a change in internal political or institutional alignment. Existing divisions over negotiations, sanctions relief, leadership uncertainty, IRGC influence and repression remain operative.
- **Conditions:**
  - Mojtaba Khamenei remains Supreme Leader but has not appeared publicly since his appointment.
  - Iranian officials remain divided over negotiations, sanctions relief and the conditions for reopening Hormuz.
  - Security forces continue arrests and repression following the 2026 protests and wartime allegations of collaboration.
  - Iranian forces have carried out the declared order to close Hormuz and attack U.S. naval forces, without establishing that broader internal divisions have been resolved.
  - Expanded U.S. strikes are targeting Iranian maritime, missile and command assets involved in enforcing the closure.

### Variable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 123
- **Index Change:** 115 → 123
- **Status:** US-Iran tension reaches a more acute level as the United States expands strikes against Iranian maritime, missile and command assets and intensifies coalition operations to defeat the closure of Hormuz.
- **Rationale:** The index rises from 115 to 123 because the event adds a broader and more direct U.S. military campaign against Iranian assets to the existing blockade, sanctions, stalled diplomacy, Iranian closure order and attacks on U.S. naval forces. The large increase reflects a major escalation in direct confrontation without asserting consequences beyond the executed operations.
- **Conditions:**
  - U.S. and Iranian forces continue exchanging limited strikes.
  - The U.S. blockade and expanded sanctions campaign remain active.
  - Diplomatic contacts have stalled, with competing accounts of possible nuclear-inspection proposals and the status of negotiations.
  - Iran has declared Hormuz closed, ordered the prevention of tanker transit and attacked U.S. naval forces operating in the Gulf.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in tanker interdiction and attacks on U.S. forces.
  - Coalition naval operations have intensified to reopen and secure the Strait.

### Variable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 118
- **Index Change:** 110 → 118
- **Status:** Regional armed conflict intensifies substantially in the Gulf as U.S. forces strike Iranian maritime, missile and command assets and coalition naval operations expand around the Strait. Existing fighting in Lebanon, Yemen, Iraq and other theaters remains operative.
- **Rationale:** The index rises from 110 to 118 because the event broadens and intensifies direct kinetic activity between U.S. and Iranian forces around a strategically important maritime chokepoint, including strikes on multiple categories of Iranian military assets. The increase is substantial but limited because the event does not itself establish new fighting in every existing regional theater.
- **Conditions:**
  - U.S.-Iran hostilities and attacks on shipping around Hormuz remain active, with Iran declaring the Strait closed and attacking U.S. naval forces.
  - Israeli operations and Hezbollah-related conflict continue in Lebanon.
  - Houthi attacks and Saudi responses continue around Yemen and the Bab el-Mandeb.
  - Iran-aligned groups and regional forces remain involved in attacks affecting Iraq and Gulf energy infrastructure.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in attacks and tanker interdiction.
  - Coalition naval operations have intensified around the Strait to suppress Iranian enforcement of the closure.

---

# Event E2

## Event Information

- **Event ID:** E2
- **Actor:** Iran
- **Action:** Escalate asymmetric pressure by sustaining the Hormuz closure and expanding coordinated missile, drone, maritime, and partner-group attacks against U.S. and coalition military assets involved in the blockade and reopening operations.
- **Rationale:** Iran could seek to raise the operational and political costs of the expanded U.S. campaign, restore deterrence, preserve control over the escalation initiative, and demonstrate that strikes against Iranian assets will not compel immediate capitulation.
- **Selection Weight:** 0.72
- **Source WorldState:** W2
- **Resulting WorldState:** W3

## Resulting WorldState: W3

### Variable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 126
- **Index Change:** 120 → 126
- **Status:** Brent crude prices rise further as Iran sustains the Hormuz closure and expands coordinated attacks against U.S. and coalition forces involved in reopening operations, increasing the immediate risk to Gulf supply and shipping.
- **Rationale:** The index rises from 120 to 126 because the event sustains the existing closure of the principal oil-transit chokepoint and expands attacks against forces attempting to restore navigation. More than 10 million barrels per day of Gulf output remain shut in, inventories are depleted, and existing Bab el-Mandeb and Hormuz disruptions continue. The increase is substantial but limited because the event does not establish a new permanent loss of all regional production.
- **Conditions:**
  - More than 10 million barrels per day of Gulf output remained shut in during August, with recovery deferred.
  - Global observed oil inventories have fallen substantially since February, reducing available market buffers.
  - Strait of Hormuz and Bab el-Mandeb disruptions continue to constrain exports and increase shipping costs.
  - Iran continues to declare the Strait of Hormuz closed to commercial shipping and sustains efforts to prevent tanker transit.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition forces involved in blockade and reopening operations.

### Variable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 54
- **Index Change:** 62 → 54
- **Status:** Oil and petroleum-product flows through the Strait of Hormuz decline further as Iran sustains the closure and expands missile, drone, maritime and partner-group attacks against forces supporting reopening operations.
- **Rationale:** The index falls from 62 to 54 because the event directly sustains the closure order and adds coordinated attacks against military assets involved in blockade and reopening operations, increasing hazards for commercial transit and reducing the prospect of near-term normalized flows. Existing alternative pipelines, overland routes and ship-to-ship transfers prevent the index from falling further.
- **Conditions:**
  - Iranian threats, attacks and the declared closure of the Strait remain operative.
  - Iranian forces continue preventing commercial tanker transit.
  - A U.S. naval blockade and military escorts affect shipping through and around the Strait.
  - Alternative pipelines, overland routes and ship-to-ship transfers are partially offsetting the disruption.
  - Coalition naval operations continue to reopen and secure the Strait.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets involved in blockade and reopening operations.

### Variable: India Gulf Oil Imports

- **Variable Name:** India Gulf Oil Imports
- **Index:** 86
- **Index Change:** 90 → 86
- **Status:** India's Gulf-origin crude imports are further constrained as Iran sustains the Hormuz closure and expands attacks affecting military protection and commercial transit. Diversified and non-Hormuz sourcing continues to moderate the reduction.
- **Rationale:** The index falls from 90 to 86 because sustained closure and expanded attacks against coalition forces further reduce the reliability of Gulf-origin crude shipments reaching India. India's sourcing from approximately 40 countries and use of alternative maritime routes limit the decline, but they do not remove the effect of lower Hormuz flows and greater shipping risk.
- **Conditions:**
  - India remains dependent on imported oil, with reported oil-and-gas import dependency of approximately 88.7%.
  - India is sourcing crude from approximately 40 countries and using alternative maritime routes.
  - Hormuz and Gulf shipping disruptions continue to affect the reliability and composition of Indian crude supplies.
  - Iran continues its closure order and tanker-prevention operations, restricting Gulf-origin crude reaching India through Hormuz.
  - Expanded Iranian missile, drone, maritime and partner-group attacks create additional hazards for Gulf-origin shipping and its military protection.

### Variable: Iran Sanctions Enforcement

- **Variable Name:** Iran Sanctions Enforcement
- **Index:** 100
- **Index Change:** 100 → 100
- **Status:** Sanctions enforcement remains extensive but is not directly changed by Iran's expanded military and partner-group attacks. Existing enforcement pressure and Iranian circumvention networks remain operative.
- **Rationale:** The index remains at 100 because the event expands military and asymmetric pressure without changing sanctions authorities, financial enforcement mechanisms or the established effectiveness of Iranian circumvention networks. The existing blockade, sanctions campaign and shadow-fleet and financial networks therefore remain the relevant conditions.
- **Conditions:**
  - The U.S. blockade of Iranian ports remains in place.
  - The United States is conducting an expanded sanctions campaign targeting oil, shipping, finance and additional economic sectors.
  - Iran continues to rely on shadow-fleet and Russia-linked financial networks to facilitate trade and sanctions evasion.

### Variable: US/Allied Cohesion

- **Variable Name:** US/Allied Cohesion
- **Index:** 110
- **Index Change:** 108 → 110
- **Status:** U.S. and cooperating states face expanded Iranian missile, drone, maritime and partner-group attacks while continuing naval coordination around the Strait. The shared security pressure strengthens operational alignment, while existing disagreements remain.
- **Rationale:** The index rises from 108 to 110 because Iran's expanded attacks directly affect U.S. and coalition military assets and create an additional shared operational threat around the closure and reopening effort. The increase is limited because the event does not resolve existing differences over military access, retaliation risk, sanctions and diplomatic policy.
- **Conditions:**
  - U.S. and Israeli military operations and strategic objectives remain linked.
  - Gulf states face continuing Iranian or Iran-linked attacks and are assessing security cooperation with the United States.
  - Regional governments are pursuing parallel diplomatic engagement while weighing the costs of confrontation.
  - Iran's closure of Hormuz and attacks on U.S. naval forces create immediate security pressure on the United States and cooperating states.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets.
  - Coalition naval operations continue to suppress Iranian tanker interdiction and reopen and secure the Strait.

### Variable: Iran Internal Cohesion

- **Variable Name:** Iran Internal Cohesion
- **Index:** 100
- **Index Change:** 100 → 100
- **Status:** Iran's governing and security apparatus sustains the closure and expands coordinated attacks while facing continued external strikes. The event provides no direct evidence that underlying political divisions, leadership uncertainty or IRGC influence have materially changed.
- **Rationale:** The index remains at 100 because the executed action demonstrates continued coordination of Iranian and partner-group operations but does not establish a broader change in internal political or institutional alignment. Existing divisions over negotiations, sanctions relief, leadership uncertainty, IRGC influence and repression remain operative.
- **Conditions:**
  - Mojtaba Khamenei remains Supreme Leader but has not appeared publicly since his appointment.
  - Iranian officials remain divided over negotiations, sanctions relief and the conditions for reopening Hormuz.
  - Security forces continue arrests and repression following the 2026 protests and wartime allegations of collaboration.
  - Iranian forces continue enforcing the closure of Hormuz and attacking U.S. and coalition military assets, without establishing that broader internal divisions have been resolved.
  - Iran sustains coordinated missile, drone, maritime and partner-group operations against forces involved in the blockade and reopening effort.

### Variable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 132
- **Index Change:** 123 → 132
- **Status:** US-Iran tension reaches a more acute level as Iran sustains the Hormuz closure and expands coordinated attacks against U.S. and coalition military assets, while existing blockade, sanctions and direct hostilities continue.
- **Rationale:** The index rises from 123 to 132 because the event adds sustained and broader asymmetric attacks by Iran to an already active confrontation involving U.S. strikes, the blockade, sanctions, stalled diplomacy, the closure order and attacks on naval forces. The large increase reflects intensified direct and indirect confrontation without assuming effects beyond the executed operations.
- **Conditions:**
  - U.S. and Iranian forces continue exchanging limited strikes.
  - The U.S. blockade and expanded sanctions campaign remain active.
  - Diplomatic contacts have stalled, with competing accounts of possible nuclear-inspection proposals and the status of negotiations.
  - Iran continues declaring Hormuz closed, preventing tanker transit and attacking U.S. naval forces operating in the Gulf.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in tanker interdiction and attacks on U.S. forces.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets involved in blockade and reopening operations.
  - Coalition naval operations continue to reopen and secure the Strait.

### Variable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 126
- **Index Change:** 118 → 126
- **Status:** Regional armed conflict intensifies as Iran sustains the Hormuz closure and expands coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition assets. Existing fighting in Lebanon, Yemen, Iraq and other theaters remains operative.
- **Rationale:** The index rises from 118 to 126 because the event expands the scale and coordination of kinetic activity around the Gulf, involving missile, drone, maritime and partner-group attacks against forces engaged in the blockade and reopening operations. The increase is substantial but limited because the event does not itself establish new fighting in every existing regional theater.
- **Conditions:**
  - U.S.-Iran hostilities and attacks on shipping around Hormuz remain active, with Iran declaring the Strait closed and attacking U.S. naval forces.
  - Israeli operations and Hezbollah-related conflict continue in Lebanon.
  - Houthi attacks and Saudi responses continue around Yemen and the Bab el-Mandeb.
  - Iran-aligned groups and regional forces remain involved in attacks affecting Iraq and Gulf energy infrastructure.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in attacks and tanker interdiction.
  - Coalition naval operations continue around the Strait to suppress Iranian enforcement of the closure.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets.

In [27]:


display(Markdown(StructuredReport))

# Scenario Summary

- **Number of Events:** 3
- **Number of WorldStates:** 4
- **Initial WorldState:** W0
- **Terminal WorldState:** W3

## WorldState Sequence

| WorldState | Preceding Event | Index Changes |
|---|---|---|
| W0 | Initial state | Baseline |
| W1 | E0 | Oil Price 100→115; Hormuz Oil Flow 100→65; India Gulf Oil Imports 100→92; Iran Sanctions Enforcement 100→100; US/Allied Cohesion 100→104; Iran Internal Cohesion 100→100; US-Iran Tension 100→115; Regional Armed Conflict 100→110 |
| W2 | E1 | Oil Price 115→120; Hormuz Oil Flow 65→62; India Gulf Oil Imports 92→90; Iran Sanctions Enforcement 100→100; US/Allied Cohesion 104→108; Iran Internal Cohesion 100→100; US-Iran Tension 115→123; Regional Armed Conflict 110→118 |
| W3 | E2 | Oil Price 120→126; Hormuz Oil Flow 62→54; India Gulf Oil Imports 90→86; Iran Sanctions Enforcement 100→100; US/Allied Cohesion 108→110; Iran Internal Cohesion 100→100; US-Iran Tension 123→132; Regional Armed Conflict 118→126 |

# Event E0

- **Event ID:** E0
- **Actor:** Iran
- **Action:** Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
- **Rationale:** Initial Trigger for Iran
- **Selection Weight:** None
- **Source WorldState:** W0
- **Resulting WorldState:** W1

## Resulting WorldState W1

### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 115
- **Change from Preceding WorldState:** 100→115
- **Status:** Brent crude prices rise further as Iran's declared closure of the Strait of Hormuz and attacks on U.S. naval forces intensify the existing Gulf supply and shipping disruption.
- **Rationale:** The index rises from 100 to 115 because the event directly threatens a major oil-transit route already affected by more than 10 million barrels per day of shut-in Gulf output, depleted inventories and maritime disruption. The additional closure order and attacks justify a substantial but not extreme increase above the already elevated baseline condition.
- **Conditions:**
  - More than 10 million barrels per day of Gulf output remained shut in during August, with recovery deferred.
  - Global observed oil inventories have fallen substantially since February, reducing available market buffers.
  - Strait of Hormuz and Bab el-Mandeb disruptions continue to constrain exports and increase shipping costs.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.

### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 65
- **Change from Preceding WorldState:** 100→65
- **Status:** Oil and petroleum-product flows through the Strait of Hormuz are reduced to a severely restricted level. Iran's closure order and efforts to prevent tanker transit further limit the previously existing escorted, bypass and workaround movements.
- **Rationale:** The index falls from 100 to 65 because the event directly orders the prevention of commercial tanker transit, worsening a condition that was already severely constrained by Iranian attacks, U.S. naval operations and limited workarounds. Some residual movement may remain through escorts, alternative routes or transfers, so the flow is not assessed as zero.
- **Conditions:**
  - Iranian threats, attacks and the declared closure of the Strait remain operative.
  - Iranian forces have been ordered to prevent commercial tanker transit.
  - A U.S. naval blockade and military escorts affect shipping through and around the Strait.
  - Alternative pipelines, overland routes and ship-to-ship transfers are partially offsetting the disruption.

### WorldVariable: India Gulf Oil Imports

- **Variable Name:** India Gulf Oil Imports
- **Index:** 92
- **Change from Preceding WorldState:** 100→92
- **Status:** India's imports of Gulf-origin crude are further constrained by the declared closure of the Strait of Hormuz and attacks on naval forces. India continues to use diversified and non-Hormuz sourcing, but the reliability and volume of Gulf-linked imports are reduced.
- **Rationale:** The index falls from 100 to 92 because the event directly restricts the principal maritime route affecting Gulf-origin supplies, adding to existing reliability problems. India's prior diversification, sourcing from approximately 40 countries and use of alternative routes moderate the decline rather than eliminating it.
- **Conditions:**
  - India remains dependent on imported oil, with reported oil-and-gas import dependency of approximately 88.7%.
  - India is sourcing crude from approximately 40 countries and using alternative maritime routes.
  - Hormuz and Gulf shipping disruptions continue to affect the reliability and composition of Indian crude supplies.
  - Iran's closure order and tanker-prevention operations further restrict Gulf-origin crude reaching India through Hormuz.

### WorldVariable: Iran Sanctions Enforcement

- **Variable Name:** Iran Sanctions Enforcement
- **Index:** 100
- **Change from Preceding WorldState:** 100→100
- **Status:** Sanctions enforcement remains extensive but is not directly altered by Iran's operational closure of the Strait and attacks on U.S. naval forces. Existing enforcement pressure and Iranian circumvention networks remain operative.
- **Rationale:** The index remains at 100 because the event changes maritime and military conditions but does not itself establish a change in the effectiveness of sanctions enforcement. The existing U.S. blockade, expanded sanctions campaign and Iranian shadow-fleet and financial networks continue to define this variable.
- **Conditions:**
  - The U.S. blockade of Iranian ports remains in place.
  - The United States is conducting an expanded sanctions campaign targeting oil, shipping, finance and additional economic sectors.
  - Iran continues to rely on shadow-fleet and Russia-linked financial networks to facilitate trade and sanctions evasion.

### WorldVariable: US/Allied Cohesion

- **Variable Name:** US/Allied Cohesion
- **Index:** 104
- **Change from Preceding WorldState:** 100→104
- **Status:** The Iranian closure declaration and attacks on U.S. naval forces create an immediate common security pressure for the United States and cooperating states. Alignment remains incomplete because partners continue to differ over military access, exposure to retaliation, sanctions and diplomacy.
- **Rationale:** The index rises from 100 to 104 because a direct Iranian challenge to commercial shipping and U.S. naval forces provides additional grounds for coordinated security responses among existing partners. The increase is limited by the operative disagreements over access, retaliation risk, sanctions and diplomatic policy.
- **Conditions:**
  - U.S. and Israeli military operations and strategic objectives remain linked.
  - Gulf states face continuing Iranian or Iran-linked attacks and are assessing security cooperation with the United States.
  - Regional governments are pursuing parallel diplomatic engagement while weighing the costs of confrontation.
  - Iran's closure of Hormuz and attacks on U.S. naval forces create additional immediate security pressure on the United States and cooperating states.

### WorldVariable: Iran Internal Cohesion

- **Variable Name:** Iran Internal Cohesion
- **Index:** 100
- **Change from Preceding WorldState:** 100→100
- **Status:** Iran's governing and security apparatus has executed a unified closure and attack order, but the event provides no direct evidence that the underlying political divisions, leadership uncertainty or IRGC influence have materially changed.
- **Rationale:** The index remains at 100 because execution of the stated military action demonstrates operational implementation but does not by itself establish a broader change in internal political cohesion. Existing divisions over negotiations, leadership uncertainty, IRGC influence and repression remain operative.
- **Conditions:**
  - Mojtaba Khamenei remains Supreme Leader but has not appeared publicly since his appointment.
  - Iranian officials remain divided over negotiations, sanctions relief and the conditions for reopening Hormuz.
  - Security forces continue arrests and repression following the 2026 protests and wartime allegations of collaboration.
  - Iranian forces have carried out the declared order to close Hormuz and attack U.S. naval forces, without establishing that broader internal divisions have been resolved.

### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 115
- **Change from Preceding WorldState:** 100→115
- **Status:** US-Iran tension increases to an even more acute level as Iran declares Hormuz closed, attempts to prevent tanker transit and attacks U.S. naval forces amid an existing blockade, sanctions campaign and stalled diplomacy.
- **Rationale:** The index rises from 100 to 115 because the event adds direct Iranian attacks on U.S. military forces and an explicit challenge to commercial navigation to the already active blockade, sanctions and limited strikes. The magnitude reflects a major escalation in military confrontation without asserting a broader change beyond the event.
- **Conditions:**
  - U.S. and Iranian forces continue exchanging limited strikes.
  - The U.S. blockade and expanded sanctions campaign remain active.
  - Diplomatic contacts have stalled, with competing accounts of possible nuclear-inspection proposals and the status of negotiations.
  - Iran has declared Hormuz closed, ordered the prevention of tanker transit and attacked U.S. naval forces operating in the Gulf.

### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 110
- **Change from Preceding WorldState:** 100→110
- **Status:** Regional armed conflict intensifies in the Gulf as Iranian forces attack U.S. naval forces while attempting to enforce closure of the Strait. Existing fighting in Lebanon, Yemen, Iraq and other theaters remains operative.
- **Rationale:** The index rises from 100 to 110 because the event adds direct kinetic attacks on U.S. naval forces and increases military activity around a strategically important maritime chokepoint. The increase is significant but limited because the event does not itself establish new fighting in every existing theater.
- **Conditions:**
  - U.S.-Iran hostilities and attacks on shipping around Hormuz remain active, with Iran now declaring the Strait closed and attacking U.S. naval forces.
  - Israeli operations and Hezbollah-related conflict continue in Lebanon.
  - Houthi attacks and Saudi responses continue around Yemen and the Bab el-Mandeb.
  - Iran-aligned groups and regional forces remain involved in attacks affecting Iraq and Gulf energy infrastructure.

# Event E1

- **Event ID:** E1
- **Actor:** US
- **Action:** Expand military operations to suppress Iran's ability to enforce the closure, including strikes against Iranian maritime, missile and command assets involved in attacks on U.S. forces and tanker interdiction, while intensifying coalition naval operations to reopen and secure the Strait.
- **Rationale:** A forceful response could restore deterrence, protect U.S. forces and commercial shipping, impose immediate costs for the attacks, and demonstrate that Iran cannot gain strategic leverage by closing Hormuz.
- **Selection Weight:** 0.8
- **Source WorldState:** W1
- **Resulting WorldState:** W2

## Resulting WorldState W2

### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 120
- **Change from Preceding WorldState:** 115→120
- **Status:** Brent crude prices rise further as expanded U.S. strikes and intensified coalition operations add immediate military risk around the already closed Strait of Hormuz and Gulf shipping routes.
- **Rationale:** The index rises from 115 to 120 because the event expands direct military operations around the principal oil-transit chokepoint while more than 10 million barrels per day of Gulf output remain shut in and inventories are depleted. The increase is substantial but limited because the operations are also intended to restore navigation rather than establish a permanent additional supply loss.
- **Conditions:**
  - More than 10 million barrels per day of Gulf output remained shut in during August, with recovery deferred.
  - Global observed oil inventories have fallen substantially since February, reducing available market buffers.
  - Strait of Hormuz and Bab el-Mandeb disruptions continue to constrain exports and increase shipping costs.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - Expanded U.S. strikes and coalition naval operations are creating additional immediate military risk around the Strait and Gulf shipping.

### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 62
- **Change from Preceding WorldState:** 65→62
- **Status:** Oil and petroleum-product flows through the Strait of Hormuz fall to an even more restricted level during expanded U.S. strikes and intensified coalition operations. The operations may support eventual reopening, but the immediate combat environment further impedes commercial transit.
- **Rationale:** The index falls from 65 to 62 because expanded strikes against maritime, missile and command assets involved in tanker interdiction increase immediate combat hazards around a route already subject to Iran's closure order and U.S. naval operations. The decline is modest because intensified coalition operations also preserve some potential for escorted or controlled movement and eventual reopening.
- **Conditions:**
  - Iranian threats, attacks and the declared closure of the Strait remain operative.
  - Iranian forces have been ordered to prevent commercial tanker transit.
  - A U.S. naval blockade and military escorts affect shipping through and around the Strait.
  - Alternative pipelines, overland routes and ship-to-ship transfers are partially offsetting the disruption.
  - Expanded U.S. strikes target Iranian maritime, missile and command assets involved in tanker interdiction.
  - Coalition naval operations have intensified to reopen and secure the Strait.

### WorldVariable: India Gulf Oil Imports

- **Variable Name:** India Gulf Oil Imports
- **Index:** 90
- **Change from Preceding WorldState:** 92→90
- **Status:** India's Gulf-origin crude imports are further constrained by intensified military operations around the closed Strait of Hormuz. Diversified and non-Hormuz sourcing continues to moderate, but does not eliminate, the reduction in Gulf-linked supply reliability.
- **Rationale:** The index falls from 92 to 90 because expanded strikes and intensified naval operations add immediate transit risk to an already restricted route, further reducing the reliability of Gulf-origin crude reaching India. India's sourcing from approximately 40 countries and use of alternative routes limit the magnitude of the decline.
- **Conditions:**
  - India remains dependent on imported oil, with reported oil-and-gas import dependency of approximately 88.7%.
  - India is sourcing crude from approximately 40 countries and using alternative maritime routes.
  - Hormuz and Gulf shipping disruptions continue to affect the reliability and composition of Indian crude supplies.
  - Iran's closure order and tanker-prevention operations further restrict Gulf-origin crude reaching India through Hormuz.
  - Expanded U.S. and coalition military operations create additional short-term hazards for Gulf-origin shipping.

### WorldVariable: Iran Sanctions Enforcement

- **Variable Name:** Iran Sanctions Enforcement
- **Index:** 100
- **Change from Preceding WorldState:** 100→100
- **Status:** Sanctions enforcement remains extensive but is not directly changed by the expansion of military operations against Iranian maritime, missile and command assets. Existing enforcement pressure and Iranian circumvention networks remain operative.
- **Rationale:** The index remains at 100 because the event concerns military suppression of Iran's maritime and military capabilities rather than a change in sanctions authorities, financial enforcement or the effectiveness of Iranian circumvention networks. The existing blockade, sanctions campaign and shadow-fleet and financial networks therefore remain the relevant conditions.
- **Conditions:**
  - The U.S. blockade of Iranian ports remains in place.
  - The United States is conducting an expanded sanctions campaign targeting oil, shipping, finance and additional economic sectors.
  - Iran continues to rely on shadow-fleet and Russia-linked financial networks to facilitate trade and sanctions evasion.

### WorldVariable: US/Allied Cohesion

- **Variable Name:** US/Allied Cohesion
- **Index:** 108
- **Change from Preceding WorldState:** 104→108
- **Status:** U.S. and cooperating states intensify naval coordination to suppress Iran's interdiction efforts and secure the Strait. Existing differences over military access, exposure to retaliation, sanctions and diplomacy remain, limiting the increase in cohesion.
- **Rationale:** The index rises from 104 to 108 because the event involves intensified coalition naval operations with a shared operational purpose: reopening and securing the Strait against Iranian interdiction. The increase is limited because the event does not resolve the existing disagreements over access, retaliation risk, sanctions and diplomatic policy.
- **Conditions:**
  - U.S. and Israeli military operations and strategic objectives remain linked.
  - Gulf states face continuing Iranian or Iran-linked attacks and are assessing security cooperation with the United States.
  - Regional governments are pursuing parallel diplomatic engagement while weighing the costs of confrontation.
  - Iran's closure of Hormuz and attacks on U.S. naval forces create additional immediate security pressure on the United States and cooperating states.
  - Coalition naval operations have intensified to suppress Iranian tanker interdiction and reopen and secure the Strait.

### WorldVariable: Iran Internal Cohesion

- **Variable Name:** Iran Internal Cohesion
- **Index:** 100
- **Change from Preceding WorldState:** 100→100
- **Status:** Iran's governing and security apparatus remains engaged in enforcing the closure while facing expanded U.S. strikes. The event provides no direct evidence that underlying political divisions, leadership uncertainty or IRGC influence have materially changed.
- **Rationale:** The index remains at 100 because strikes against Iranian military and command assets establish external pressure and possible operational disruption, but do not by themselves demonstrate a change in internal political or institutional alignment. Existing divisions over negotiations, sanctions relief, leadership uncertainty, IRGC influence and repression remain operative.
- **Conditions:**
  - Mojtaba Khamenei remains Supreme Leader but has not appeared publicly since his appointment.
  - Iranian officials remain divided over negotiations, sanctions relief and the conditions for reopening Hormuz.
  - Security forces continue arrests and repression following the 2026 protests and wartime allegations of collaboration.
  - Iranian forces have carried out the declared order to close Hormuz and attack U.S. naval forces, without establishing that broader internal divisions have been resolved.
  - Expanded U.S. strikes are targeting Iranian maritime, missile and command assets involved in enforcing the closure.

### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 123
- **Change from Preceding WorldState:** 115→123
- **Status:** US-Iran tension reaches a more acute level as the United States expands strikes against Iranian maritime, missile and command assets and intensifies coalition operations to defeat the closure of Hormuz.
- **Rationale:** The index rises from 115 to 123 because the event adds a broader and more direct U.S. military campaign against Iranian assets to the existing blockade, sanctions, stalled diplomacy, Iranian closure order and attacks on U.S. naval forces. The large increase reflects a major escalation in direct confrontation without asserting consequences beyond the executed operations.
- **Conditions:**
  - U.S. and Iranian forces continue exchanging limited strikes.
  - The U.S. blockade and expanded sanctions campaign remain active.
  - Diplomatic contacts have stalled, with competing accounts of possible nuclear-inspection proposals and the status of negotiations.
  - Iran has declared Hormuz closed, ordered the prevention of tanker transit and attacked U.S. naval forces operating in the Gulf.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in tanker interdiction and attacks on U.S. forces.
  - Coalition naval operations have intensified to reopen and secure the Strait.

### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 118
- **Change from Preceding WorldState:** 110→118
- **Status:** Regional armed conflict intensifies substantially in the Gulf as U.S. forces strike Iranian maritime, missile and command assets and coalition naval operations expand around the Strait. Existing fighting in Lebanon, Yemen, Iraq and other theaters remains operative.
- **Rationale:** The index rises from 110 to 118 because the event broadens and intensifies direct kinetic activity between U.S. and Iranian forces around a strategically important maritime chokepoint, including strikes on multiple categories of Iranian military assets. The increase is substantial but limited because the event does not itself establish new fighting in every existing regional theater.
- **Conditions:**
  - U.S.-Iran hostilities and attacks on shipping around Hormuz remain active, with Iran declaring the Strait closed and attacking U.S. naval forces.
  - Israeli operations and Hezbollah-related conflict continue in Lebanon.
  - Houthi attacks and Saudi responses continue around Yemen and the Bab el-Mandeb.
  - Iran-aligned groups and regional forces remain involved in attacks affecting Iraq and Gulf energy infrastructure.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in attacks and tanker interdiction.
  - Coalition naval operations have intensified around the Strait to suppress Iranian enforcement of the closure.

# Event E2

- **Event ID:** E2
- **Actor:** Iran
- **Action:** Escalate asymmetric pressure by sustaining the Hormuz closure and expanding coordinated missile, drone, maritime, and partner-group attacks against U.S. and coalition military assets involved in the blockade and reopening operations.
- **Rationale:** Iran could seek to raise the operational and political costs of the expanded U.S. campaign, restore deterrence, preserve control over the escalation initiative, and demonstrate that strikes against Iranian assets will not compel immediate capitulation.
- **Selection Weight:** 0.72
- **Source WorldState:** W2
- **Resulting WorldState:** W3

## Resulting WorldState W3

### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 126
- **Change from Preceding WorldState:** 120→126
- **Status:** Brent crude prices rise further as Iran sustains the Hormuz closure and expands coordinated attacks against U.S. and coalition forces involved in reopening operations, increasing the immediate risk to Gulf supply and shipping.
- **Rationale:** The index rises from 120 to 126 because the event sustains the existing closure of the principal oil-transit chokepoint and expands attacks against forces attempting to restore navigation. More than 10 million barrels per day of Gulf output remain shut in, inventories are depleted, and existing Bab el-Mandeb and Hormuz disruptions continue. The increase is substantial but limited because the event does not establish a new permanent loss of all regional production.
- **Conditions:**
  - More than 10 million barrels per day of Gulf output remained shut in during August, with recovery deferred.
  - Global observed oil inventories have fallen substantially since February, reducing available market buffers.
  - Strait of Hormuz and Bab el-Mandeb disruptions continue to constrain exports and increase shipping costs.
  - Iran continues to declare the Strait of Hormuz closed to commercial shipping and sustains efforts to prevent tanker transit.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition forces involved in blockade and reopening operations.

### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 54
- **Change from Preceding WorldState:** 62→54
- **Status:** Oil and petroleum-product flows through the Strait of Hormuz decline further as Iran sustains the closure and expands missile, drone, maritime and partner-group attacks against forces supporting reopening operations.
- **Rationale:** The index falls from 62 to 54 because the event directly sustains the closure order and adds coordinated attacks against military assets involved in blockade and reopening operations, increasing hazards for commercial transit and reducing the prospect of near-term normalized flows. Existing alternative pipelines, overland routes and ship-to-ship transfers prevent the index from falling further.
- **Conditions:**
  - Iranian threats, attacks and the declared closure of the Strait remain operative.
  - Iranian forces continue preventing commercial tanker transit.
  - A U.S. naval blockade and military escorts affect shipping through and around the Strait.
  - Alternative pipelines, overland routes and ship-to-ship transfers are partially offsetting the disruption.
  - Coalition naval operations continue to reopen and secure the Strait.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets involved in blockade and reopening operations.

### WorldVariable: India Gulf Oil Imports

- **Variable Name:** India Gulf Oil Imports
- **Index:** 86
- **Change from Preceding WorldState:** 90→86
- **Status:** India's Gulf-origin crude imports are further constrained as Iran sustains the Hormuz closure and expands attacks affecting military protection and commercial transit. Diversified and non-Hormuz sourcing continues to moderate the reduction.
- **Rationale:** The index falls from 90 to 86 because sustained closure and expanded attacks against coalition forces further reduce the reliability of Gulf-origin crude shipments reaching India. India's sourcing from approximately 40 countries and use of alternative maritime routes limit the decline, but they do not remove the effect of lower Hormuz flows and greater shipping risk.
- **Conditions:**
  - India remains dependent on imported oil, with reported oil-and-gas import dependency of approximately 88.7%.
  - India is sourcing crude from approximately 40 countries and using alternative maritime routes.
  - Hormuz and Gulf shipping disruptions continue to affect the reliability and composition of Indian crude supplies.
  - Iran continues its closure order and tanker-prevention operations, restricting Gulf-origin crude reaching India through Hormuz.
  - Expanded Iranian missile, drone, maritime and partner-group attacks create additional hazards for Gulf-origin shipping and its military protection.

### WorldVariable: Iran Sanctions Enforcement

- **Variable Name:** Iran Sanctions Enforcement
- **Index:** 100
- **Change from Preceding WorldState:** 100→100
- **Status:** Sanctions enforcement remains extensive but is not directly changed by Iran's expanded military and partner-group attacks. Existing enforcement pressure and Iranian circumvention networks remain operative.
- **Rationale:** The index remains at 100 because the event expands military and asymmetric pressure without changing sanctions authorities, financial enforcement mechanisms or the established effectiveness of Iranian circumvention networks. The existing blockade, sanctions campaign and shadow-fleet and financial networks therefore remain the relevant conditions.
- **Conditions:**
  - The U.S. blockade of Iranian ports remains in place.
  - The United States is conducting an expanded sanctions campaign targeting oil, shipping, finance and additional economic sectors.
  - Iran continues to rely on shadow-fleet and Russia-linked financial networks to facilitate trade and sanctions evasion.

### WorldVariable: US/Allied Cohesion

- **Variable Name:** US/Allied Cohesion
- **Index:** 110
- **Change from Preceding WorldState:** 108→110
- **Status:** U.S. and cooperating states face expanded Iranian missile, drone, maritime and partner-group attacks while continuing naval coordination around the Strait. The shared security pressure strengthens operational alignment, while existing disagreements remain.
- **Rationale:** The index rises from 108 to 110 because Iran's expanded attacks directly affect U.S. and coalition military assets and create an additional shared operational threat around the closure and reopening effort. The increase is limited because the event does not resolve existing differences over military access, retaliation risk, sanctions and diplomatic policy.
- **Conditions:**
  - U.S. and Israeli military operations and strategic objectives remain linked.
  - Gulf states face continuing Iranian or Iran-linked attacks and are assessing security cooperation with the United States.
  - Regional governments are pursuing parallel diplomatic engagement while weighing the costs of confrontation.
  - Iran's closure of Hormuz and attacks on U.S. naval forces create immediate security pressure on the United States and cooperating states.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets.
  - Coalition naval operations continue to suppress Iranian tanker interdiction and reopen and secure the Strait.

### WorldVariable: Iran Internal Cohesion

- **Variable Name:** Iran Internal Cohesion
- **Index:** 100
- **Change from Preceding WorldState:** 100→100
- **Status:** Iran's governing and security apparatus sustains the closure and expands coordinated attacks while facing continued external strikes. The event provides no direct evidence that underlying political divisions, leadership uncertainty or IRGC influence have materially changed.
- **Rationale:** The index remains at 100 because the executed action demonstrates continued coordination of Iranian and partner-group operations but does not establish a broader change in internal political or institutional alignment. Existing divisions over negotiations, sanctions relief, leadership uncertainty, IRGC influence and repression remain operative.
- **Conditions:**
  - Mojtaba Khamenei remains Supreme Leader but has not appeared publicly since his appointment.
  - Iranian officials remain divided over negotiations, sanctions relief and the conditions for reopening Hormuz.
  - Security forces continue arrests and repression following the 2026 protests and wartime allegations of collaboration.
  - Iranian forces continue enforcing the closure of Hormuz and attacking U.S. and coalition military assets, without establishing that broader internal divisions have been resolved.
  - Iran sustains coordinated missile, drone, maritime and partner-group operations against forces involved in the blockade and reopening effort.

### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 132
- **Change from Preceding WorldState:** 123→132
- **Status:** US-Iran tension reaches a more acute level as Iran sustains the Hormuz closure and expands coordinated attacks against U.S. and coalition military assets, while existing blockade, sanctions and direct hostilities continue.
- **Rationale:** The index rises from 123 to 132 because the event adds sustained and broader asymmetric attacks by Iran to an already active confrontation involving U.S. strikes, the blockade, sanctions, stalled diplomacy, the closure order and attacks on naval forces. The large increase reflects intensified direct and indirect confrontation without assuming effects beyond the executed operations.
- **Conditions:**
  - U.S. and Iranian forces continue exchanging limited strikes.
  - The U.S. blockade and expanded sanctions campaign remain active.
  - Diplomatic contacts have stalled, with competing accounts of possible nuclear-inspection proposals and the status of negotiations.
  - Iran continues declaring Hormuz closed, preventing tanker transit and attacking U.S. naval forces operating in the Gulf.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in tanker interdiction and attacks on U.S. forces.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets involved in blockade and reopening operations.
  - Coalition naval operations continue to reopen and secure the Strait.

### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 126
- **Change from Preceding WorldState:** 118→126
- **Status:** Regional armed conflict intensifies as Iran sustains the Hormuz closure and expands coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition assets. Existing fighting in Lebanon, Yemen, Iraq and other theaters remains operative.
- **Rationale:** The index rises from 118 to 126 because the event expands the scale and coordination of kinetic activity around the Gulf, involving missile, drone, maritime and partner-group attacks against forces engaged in the blockade and reopening operations. The increase is substantial but limited because the event does not itself establish new fighting in every existing regional theater.
- **Conditions:**
  - U.S.-Iran hostilities and attacks on shipping around Hormuz remain active, with Iran declaring the Strait closed and attacking U.S. naval forces.
  - Israeli operations and Hezbollah-related conflict continue in Lebanon.
  - Houthi attacks and Saudi responses continue around Yemen and the Bab el-Mandeb.
  - Iran-aligned groups and regional forces remain involved in attacks affecting Iraq and Gulf energy infrastructure.
  - The United States has expanded strikes against Iranian maritime, missile and command assets involved in attacks and tanker interdiction.
  - Coalition naval operations continue around the Strait to suppress Iranian enforcement of the closure.
  - Iran has expanded coordinated missile, drone, maritime and partner-group attacks against U.S. and coalition military assets.

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)